In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FinancialReportingETL") \
    .getOrCreate()

In [ ]:
spark_df = spark.read.csv(
    "/content/Financial Statements.csv",
    header=True,
    inferSchema=True
)

spark_df.show(5)

+----+--------+--------+--------------------+--------+------------+----------+-----------------+--------+-------------------+------------------------+------------------------+-----------------------------------+-------------+-----------------+--------+-------+-------+-----------------+------------------------+-------------------------+-------------------+---------------------+
|Year|Company |Category|Market Cap(in B USD)| Revenue|Gross Profit|Net Income|Earning Per Share|  EBITDA|Share Holder Equity|Cash Flow from Operating|Cash Flow from Investing|Cash Flow from Financial Activities|Current Ratio|Debt/Equity Ratio|     ROE|    ROA|    ROI|Net Profit Margin|Free Cash Flow per Share|Return on Tangible Equity|Number of Employees|Inflation Rate(in US)|
+----+--------+--------+--------------------+--------+------------+----------+-----------------+--------+-------------------+------------------------+------------------------+-----------------------------------+-------------+---------------

In [ ]:
from pyspark.sql.functions import col

spark_df = spark_df.withColumnRenamed("Company ", "company_name") \
                   .withColumnRenamed("Year", "report_year") \
                   .withColumnRenamed("Revenue", "total_revenue") \
                   .withColumnRenamed("Net Income", "net_income") \
                   .withColumnRenamed("Gross Profit", "gross_profit") \
                   .withColumnRenamed("EBITDA", "ebitda") \
                   .withColumnRenamed("Net Profit Margin", "profit_margin")

In [ ]:
from pyspark.sql.functions import to_date, concat, lit

spark_df = spark_df.withColumn(
    "report_date",
    to_date(concat(col("report_year"), lit("-12-31")))
)

In [ ]:
dim_company = spark_df.select("company_name").distinct()

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

dim_company = dim_company.withColumn(
    "company_id",
    monotonically_increasing_id()
)

In [ ]:
from pyspark.sql.functions import year, date_format

dim_date = spark_df.select("report_date").distinct()

dim_date = dim_date.withColumn(
    "date_id",
    date_format(col("report_date"), "yyyyMMdd")
).withColumn(
    "year",
    year(col("report_date"))
)

In [ ]:
fact_financials = spark_df.join(
    dim_company,
    on="company_name",
    how="left"
)

fact_financials = fact_financials.withColumn(
    "date_id",
    date_format(col("report_date"), "yyyyMMdd")
)

fact_financials = fact_financials.select(
    "company_id",
    "date_id",
    "total_revenue",
    "gross_profit",
    "net_income",
    "ebitda",
    "profit_margin",
    "ROE",
    "ROA",
    "ROI"
)

In [ ]:
fact_pd = fact_financials.toPandas()
dim_company_pd = dim_company.toPandas()
dim_date_pd = dim_date.toPandas()

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project="financial-reporting-af")

print("BigQuery client initialized")

BigQuery client initialized


In [ ]:
from google.cloud import bigquery

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

client.load_table_from_dataframe(
    dim_company_pd,
    "financial-reporting-af.financial_reporting.dim_company",
    job_config=job_config
).result()

client.load_table_from_dataframe(
    dim_date_pd,
    "financial-reporting-af.financial_reporting.dim_date",
    job_config=job_config
).result()

client.load_table_from_dataframe(
    fact_pd,
    "financial-reporting-af.financial_reporting.fact_financials",
    job_config=job_config
).result()

print("Warehouse updated successfully")

Warehouse updated successfully


In [ ]:
print(kaggle_df[["company_name", "report_year", "total_revenue"]].head())

  company_name  report_year  total_revenue
0         AAPL         2022       394328.0
1         AAPL         2021       365817.0
2         AAPL         2020       274515.0
3         AAPL         2019       260174.0
4         AAPL         2018       265595.0
